<a href="https://colab.research.google.com/github/Yacqueline12/tecnm-lenguajes-automatas-2/blob/main/laboratorios/sesion10/sesion10_laboratorio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lenguajes y Autómatas II (SCD-1016)
## Sesión 10 · Hito 1: Mini-Evaluador de Expresiones y AST Completo

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Puede un programa leer una expresión escrita y devolver su valor sin que nadie la evalúe antes?
>
> **Objetivo de la sesión:** Integrar lo aprendido en la unidad en un mini-evaluador que lee una expresión desde texto, respeta precedencia y paréntesis, y reporta errores de sintaxis y división entre cero.

**Entregable:** tu evaluador completo en artesano_sesion10_c.c, el harness en verde y la bitácora del hito

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

El analizador ya viene armado: solo completas primaria, termino y expr. Debes leer enteros, + - * /, paréntesis y espacios. error=1 es división entre cero; error=2, sintaxis inválida.

Al final de la sesión tendrás: **Un mini-evaluador que convierte 2+3*4 en 14, respeta paréntesis y distingue error de sintaxis de división entre cero.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 10](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion10.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion10.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion10.h"

long evaluar_expresion(const char *fuente, int *error);
```

- Sin variables ni menos unario: solo enteros y los operadores + - * /.
- Respeta precedencia y paréntesis.
- Completa las tres funciones marcadas; no cambies la firma de evaluar_expresion.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion10_c.c` junto con `artesano_sesion10_c.c`
> (o `ia_sesion10_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `"2+3*4"` | 14 |
| 2 | `"(2+3)*4"` | 20 |
| 3 | `"10/0" -> error 1` | 1 |
| 4 | `"2+" -> error 2` | 2 |
| 5 | `"  7 "` | 7 |
| 6 | `"5-10"` | -5 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 10 - Mini-evaluador de expresiones (Hito 1) ===
Caso                               esperado:              obtenido:              resultado
[ 1] "2+3*4"                            esperado: 14           obtenido: 14           OK
[ 2] "(2+3)*4"                          esperado: 20           obtenido: 20           OK
[ 3] "10/0" -> error 1                  esperado: 1            obtenido: 1            OK
[ 4] "2+" -> error 2                    esperado: 2            obtenido: 2            OK
[ 5] "  7 "                             esperado: 7            obtenido: 7            OK
[ 6] "5-10"                             esperado: -5           obtenido: -5           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion10.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] El analizador respeta precedencia: la multiplicación se resuelve antes que la suma.
- [ ] Los paréntesis y los espacios no alteran el resultado.
- [ ] La división entre cero y la sintaxis inválida se reportan con códigos distintos.


## Paso 0 · Entorno


In [19]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion10_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [20]:
%%writefile api_sesion10.h
/* api_sesion10.h - contrato de la Sesion 10
 * Mini-evaluador de expresiones (Hito 1)
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION10_H
#define API_SESION10_H

long evaluar_expresion(const char *fuente, int *error);

#endif /* API_SESION10_H */


Overwriting api_sesion10.h


In [21]:
%%writefile artesano_sesion10_c.c
/* artesano_sesion10_c.c - Fase 1: El Artesano (sin IA)
 * Mini-evaluador de expresiones (Hito 1).
 *
 * Implementa lo que declara api_sesion10.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion10.h"
#include <stdio.h>
#include <string.h>
#include <ctype.h>

typedef struct { const char *s; int error; } Parser;

static void saltar(Parser *p) {
    while (*p->s == ' ') p->s++;
}

long expr(Parser *p);

long primaria(Parser *p) {
    long valor = 0;

    saltar(p);

    if (*p->s == '(') {
        p->s++;
        valor = expr(p);

        saltar(p);

        if (*p->s == ')') {
            p->s++;
        } else {
            p->error = 2;
        }

        return valor;
    }

    if (isdigit((unsigned char)*p->s)) {
        while (isdigit((unsigned char)*p->s)) {
            valor = valor * 10 + (*p->s - '0');
            p->s++;
        }
        return valor;
    }

    p->error = 2;
    return 0;
}

long termino(Parser *p) {
    long valor = primaria(p);

    while (p->error == 0) {
        long derecho;

        saltar(p);

        if (*p->s == '*') {
            p->s++;
            derecho = primaria(p);
            valor *= derecho;
        }
        else if (*p->s == '/') {
            p->s++;
            derecho = primaria(p);

            if (derecho == 0) {
                p->error = 1;
                return 0;
            }

            valor /= derecho;
        }
        else {
            break;
        }
    }

    return valor;
}

long expr(Parser *p) {
    long valor = termino(p);

    while (p->error == 0) {
        long derecho;

        saltar(p);

        if (*p->s == '+') {
            p->s++;
            derecho = termino(p);
            valor += derecho;
        }
        else if (*p->s == '-') {
            p->s++;
            derecho = termino(p);
            valor -= derecho;
        }
        else {
            break;
        }
    }

    return valor;
}

long evaluar_expresion(const char *fuente, int *error) {
    static int sin_error;
    if (error == NULL) error = &sin_error;
    *error = 0;
    if (fuente == NULL) { *error = 2; return 0; }
    Parser p = { fuente, 0 };
    long v = expr(&p);
    saltar(&p);
    if (p.error == 0 && *p.s != '\0') p.error = 2;
    *error = p.error;
    return p.error != 0 ? 0 : v;
}

Overwriting artesano_sesion10_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion10_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


Contexto: estoy realizando una práctica de Lenguajes y Autómatas II donde necesito crear un mini evaluador de expresiones matemáticas. El programa debe leer una expresión escrita como texto y calcular su resultado respetando la precedencia de operadores y el uso de paréntesis.

Rol: actúa como un programador con experiencia en lenguaje C y en la construcción de analizadores sintácticos sencillos.

Especificación: necesito implementar la función:long evaluar_expresion(const char *fuente, int *error); La solución debe aceptar números enteros, los operadores +, -, * y /, además de paréntesis y espacios en blanco, tambien debe respetar la precedencia de operadores, evaluando primero multiplicaciones y divisiones antes que sumas y restas.
Los códigos de error son:

error = 1 para división entre cero.
error = 2 para expresiones con sintaxis inválida.

No se deben utilizar variables dentro de la expresión ni implementar menos unario. La firma original debe conservarse exactamente igual, no debe incluirse una función main() y el código debe compilar correctamente usando:
-Wall -Wextra -Werror

Output: devuelve únicamente el contenido completo de un archivo .c listo para compilar, sin explicaciones adicionales.

In [33]:
%%writefile ia_sesion10_c.c
/* ia_sesion10_c.c - Fase 2: El Cientifico
 *
 * Pega aqui el codigo que te devolvio la IA, con la MISMA firma
 * que declara api_sesion10.h. No puede traer main().
 */
#include "api_sesion10.h"
#include <ctype.h>
#include <stddef.h>

typedef struct { const char *s; int error; } Parser;

static void saltar(Parser *p)
{
    while (*p->s == ' ')
        p->s++;
}

static long leer_numero(Parser *p)
{
    long n = 0;

    while (isdigit((unsigned char)*p->s))
    {
        n = n * 10 + (*p->s - '0');
        p->s++;
    }

    return n;
}

long expr(Parser *p);

long primaria(Parser *p)
{
    long valor;

    saltar(p);

    if (isdigit((unsigned char)*p->s))
        return leer_numero(p);

    if (*p->s == '(')
    {
        p->s++;

        valor = expr(p);

        saltar(p);

        if (*p->s != ')')
        {
            p->error = 2;
            return 0;
        }

        p->s++;
        return valor;
    }

    p->error = 2;
    return 0;
}

long termino(Parser *p)
{
    long resultado = primaria(p);

    while (p->error == 0)
    {
        long rhs;

        saltar(p);

        if (*p->s == '*')
        {
            p->s++;
            rhs = primaria(p);
            resultado *= rhs;
        }
        else if (*p->s == '/')
        {
            p->s++;
            rhs = primaria(p);

            if (rhs == 0)
            {
                p->error = 1;
                return 0;
            }

            resultado /= rhs;
        }
        else
        {
            break;
        }
    }

    return resultado;
}

long expr(Parser *p)
{
    long resultado = termino(p);

    while (p->error == 0)
    {
        long rhs;

        saltar(p);

        if (*p->s == '+')
        {
            p->s++;
            rhs = termino(p);
            resultado += rhs;
        }
        else if (*p->s == '-')
        {
            p->s++;
            rhs = termino(p);
            resultado -= rhs;
        }
        else
        {
            break;
        }
    }

    return resultado;
}

long evaluar_expresion(const char *fuente, int *error)
{
    static int sin_error;

    if (error == NULL)
        error = &sin_error;

    *error = 0;

    if (fuente == NULL)
    {
        *error = 2;
        return 0;
    }

    Parser p = { fuente, 0 };

    long valor = expr(&p);

    saltar(&p);

    if (p.error == 0 && *p.s != '\0')
        p.error = 2;

    *error = p.error;

    if (p.error != 0)
        return 0;

    return valor;
}

Overwriting ia_sesion10_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [23]:
%%writefile harness_sesion10_c.c
/* harness_sesion10_c.c - auditoria automatica de la Sesion 10
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion10_c.c artesano_sesion10_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion10.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 10 - Mini-evaluador de expresiones (Hito 1) ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { int e=0; long r=evaluar_expresion("2+3*4",&e); (void)e; verificar_int("\"2+3*4\"", (long)(14), (long)(r)); }
    { int e=0; long r=evaluar_expresion("(2+3)*4",&e); (void)e; verificar_int("\"(2+3)*4\"", (long)(20), (long)(r)); }
    { int e=0; long r=evaluar_expresion("10/0",&e); (void)r; verificar_int("\"10/0\" -> error 1", (long)(1), (long)(e)); }
    { int e=0; long r=evaluar_expresion("2+",&e); (void)r; verificar_int("\"2+\" -> error 2", (long)(2), (long)(e)); }
    { int e=0; long r=evaluar_expresion("  7 ",&e); (void)e; verificar_int("\"  7 \"", (long)(7), (long)(r)); }
    { int e=0; long r=evaluar_expresion("5-10",&e); (void)e; verificar_int("\"5-10\"", (long)(-5), (long)(r)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion10.md" "\n");
    return 0;
}


Overwriting harness_sesion10_c.c


In [29]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion10_c.c artesano_sesion10_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 10 - Mini-evaluador de expresiones (Hito 1) ===
Caso                               esperado:              obtenido:              resultado
[ 1] "2+3*4"                            esperado: 14           obtenido: 14           OK
[ 2] "(2+3)*4"                          esperado: 20           obtenido: 20           OK
[ 3] "10/0" -> error 1                  esperado: 1            obtenido: 1            OK
[ 4] "2+" -> error 2                    esperado: 2            obtenido: 2            OK
[ 5] "  7 "                             esperado: 7            obtenido: 7            OK
[ 6] "5-10"                             esperado: -5           obtenido: -5           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion10.md


In [34]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion10_c.c ia_sesion10_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 10 - Mini-evaluador de expresiones (Hito 1) ===
Caso                               esperado:              obtenido:              resultado
[ 1] "2+3*4"                            esperado: 14           obtenido: 14           OK
[ 2] "(2+3)*4"                          esperado: 20           obtenido: 20           OK
[ 3] "10/0" -> error 1                  esperado: 1            obtenido: 1            OK
[ 4] "2+" -> error 2                    esperado: 2            obtenido: 2            OK
[ 5] "  7 "                             esperado: 7            obtenido: 7            OK
[ 6] "5-10"                             esperado: -5           obtenido: -5           OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion10.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion10.md` a `auditorias/auditoria_sesion10.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
